# 06 - MCP Client: descubrir y llamar capacidades

Objetivo: simular el rol del client antes de usar `ClientSession` real.


## Grafico guia

```mermaid
sequenceDiagram
    participant Host
    participant Client as MCP Client
    participant Server as MCP Server
    Host->>Client: iniciar sesion
    Client->>Server: initialize
    Client->>Server: list_tools
    Client->>Server: call_tool
    Server-->>Client: result
    Client-->>Host: respuesta
```


In [ ]:
class MockGitHubMCPServer:
    def __init__(self):
        self.tools = {
            "github_create_repo": lambda args: {"full_name": f"demo/{args['name']}", "private": args.get("private", True)},
            "github_upsert_file": lambda args: {"path": args["path"], "commit_sha": "abc123"},
        }
        self.resources = {"github://config": {"token_configured": True, "default_private_repos": True}}
        self.prompts = {"repo_bootstrap_prompt": "Create repo {project_name} for {goal}"}

    def list_tools(self):
        return list(self.tools.keys())

    def list_resources(self):
        return list(self.resources.keys())

    def list_prompts(self):
        return list(self.prompts.keys())

    def call_tool(self, name, args):
        if name not in self.tools:
            return {"error": "unknown_tool"}
        return self.tools[name](args)

    def read_resource(self, uri):
        return self.resources.get(uri, {"error": "resource_not_found"})

server = MockGitHubMCPServer()
server.list_tools(), server.list_resources(), server.list_prompts()


In [ ]:
config = server.read_resource("github://config")
repo = server.call_tool("github_create_repo", {"name": "aem4-demo", "description": "demo", "private": True})
file = server.call_tool("github_upsert_file", {"owner": "demo", "repo": "aem4-demo", "path": "README.md", "content": "# Demo"})
config, repo, file


## Donde se ve despues

En `e02_openai_host_usa_mcp_github.py`, el client real hace `list_tools()` y `call_tool()` contra el server por STDIO.
